# MedGemma + Unsloth QLoRA Diagnostic / Stop-Decision Notebook — v34

This notebook keeps the **Unsloth** path, but treats it as a diagnostic rather than a final training notebook. Its purpose is to document why the Unsloth MedGemma 4-bit route was **not moved forward** for the final project.

Project target: fine-tune/adapt MedGemma for 9-class NCT-CRC-HE-100K colorectal histology image classification, where each sample is an H&E image patch and the model should return exactly one class digit.

## Key stop criterion

Before training, the base model must produce finite logits for a simple prompt. If this fails, training is invalid because the model is numerically broken **before** LoRA, loss, optimizer, scheduler, or dataset effects.

The decisive check is:

```text
base_text_only: finite X/Y all_finite=True
```

If instead the result is:

```text
base_text_only: finite 0/Y all_finite=False, nan=Y
```

then do **not** train that mode.

## Why this notebook exists

The earlier Unsloth notebooks made progressive fixes for environment and model-loading problems:

1. Hugging Face Hub version mismatch.
2. `hf_transfer` enabled without the package installed.
3. Gated Google MedGemma repo access problems.
4. SigLIP vision tower requiring eager attention.
5. TorchAO importing APIs unavailable in the selected Torch stack.
6. TorchDynamo / Unsloth compile issues.
7. Incorrect label masking that supervised the full prompt instead of only the answer digit.
8. SigLIP vision dtype instability and gradient-checkpointing issues.

After those fixes, the remaining failure was not a training hyperparameter problem. The forward diagnostic showed the **base Unsloth-loaded MedGemma model returned all-NaN logits even for text-only input**. That ruled out the image dataset, the vision tower, LoRA adapters, digit-only labels, and the optimizer.

This notebook therefore focuses on the smallest possible proof:

- text-only forward before LoRA
- multimodal forward before LoRA
- text-only forward after LoRA
- multimodal forward after LoRA
- multiple Unsloth 4-bit repos/load modes
- optional 8-bit and 16-bit modes

If the text-only base forward is already non-finite, the Unsloth mode is not suitable for final training.

## Observed results from the completed diagnostics

The 4-bit Unsloth MedGemma modes were tested on both Kaggle **T4** and **P100**.

For both GPUs, the same failure occurred:

```text
base_text_only: finite 0/5244160 all_finite=False, nan=5244160
base_multimodal: active finite 0/262208 all_finite=False
lora_text_only: finite 0/5244160 all_finite=False
lora_multimodal: active finite 0/262208 all_finite=False
```

Interpretation:

- The model loads.
- The tokenizer/processor work.
- The diagnostic prompt is built.
- The failure occurs **inside the base model forward pass**.
- It happens before images matter.
- It happens before LoRA matters.
- It happens before the training loop matters.
- It happens on both T4 and P100.

Therefore, the current 4-bit Unsloth MedGemma path is not being moved forward as the final training path.

## How to use this notebook

Run this notebook when you need evidence for the project report or if you want to test whether a different Unsloth mode becomes stable.

Default diagnostic modes:

```python
DIAGNOSTIC_MODES_TO_RUN = [
    "4bit_unsloth_bnb_no_flag",
    "4bit_bnb_fallback",
]
```

Optional modes to test separately after restarting Kaggle:

```python
DIAGNOSTIC_MODES_TO_RUN = ["8bit_full_repo"]
```

or:

```python
DIAGNOSTIC_MODES_TO_RUN = ["16bit_full_repo"]
```

Only continue with Unsloth training if a mode gives finite logits for **base text-only forward** and **base multimodal forward**.

## 1. Dependency bootstrap for Kaggle P100 or T4


In [ ]:
# Run this cell first, before importing torch / transformers / datasets / unsloth.
# It repairs Kaggle's Python environment, validates the exact package stack,
# then restarts the kernel once so imports reload cleanly.

import os
import sys
import subprocess
from pathlib import Path
from importlib import metadata

# Kaggle can inherit HF_HUB_ENABLE_HF_TRANSFER=1 without the optional hf_transfer package.
# Disable it before any huggingface_hub import; otherwise hf_hub_download can fail before model loading.
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

# Stability flags for Kaggle + Gemma3/MedGemma vision training.
# Must be set before importing unsloth/transformers/torch where possible.
os.environ['UNSLOTH_COMPILE_DISABLE'] = '1'
os.environ['UNSLOTH_COMPILE_IGNORE_ERRORS'] = '1'
os.environ['UNSLOTH_FULLGRAPH'] = '0'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['TORCH_COMPILE_DISABLE'] = '1'

# Optional for Kaggle T4 x2 single-notebook runs. Uncomment to force one GPU.
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'

MARKER = Path('/kaggle/working/.medgemma_unsloth_deps_v33_installed')
CORE_MODULES = (
    'torch', 'numpy', 'pandas', 'scipy', 'sklearn', 'transformers',
    'datasets', 'huggingface_hub', 'tokenizers', 'peft', 'trl', 'unsloth', 'unsloth_zoo', 'torchao'
)

# Use the current Unsloth Gemma 3 notebook-compatible stack. Transformers 4.56.x
# Gemma 3 mask creation uses or_mask_function/and_mask_function, which requires
# torch >= 2.6. Pin a T4-compatible CUDA 12.4 Torch 2.6 stack.
PINNED = {
    'torch': '2.6.0',
    'torchvision': '0.21.0',
    'torchaudio': '2.6.0',
    'transformers': '4.56.2',
    'tokenizers': '0.22.1',
    'trl': '0.22.2',
    'peft': '0.18.0',
    'datasets': '4.3.0',
    'huggingface-hub': '0.36.0',
    'numpy': '1.26.4',
    'pandas': '2.2.2',
    'matplotlib': '3.8.4',
    'pillow': '11.3.0',
}


def pip_install(args):
    print('+', ' '.join(args), flush=True)
    subprocess.check_call(args)


def pkg_version(*names):
    for name in names:
        try:
            return metadata.version(name)
        except metadata.PackageNotFoundError:
            pass
    return None


def version_tuple(v):
    parts = []
    for part in v.split('+')[0].split('.')[:3]:
        m = ''.join(ch for ch in part if ch.isdigit())
        if m:
            parts.append(int(m))
    return tuple(parts)


def hub_ok(v):
    if v is None:
        return False
    t = version_tuple(v)
    return (0, 30, 0) <= t < (1, 0, 0)


def active_runtime_ok():
    try:
        import torch
        arch = torch.cuda.get_arch_list() if torch.cuda.is_available() else []
        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
        capability = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None

        versions = {
            'torch': pkg_version('torch'),
            'torchvision': pkg_version('torchvision'),
            'torchaudio': pkg_version('torchaudio'),
            'transformers': pkg_version('transformers'),
            'tokenizers': pkg_version('tokenizers'),
            'trl': pkg_version('trl'),
            'peft': pkg_version('peft'),
            'datasets': pkg_version('datasets'),
            'huggingface-hub': pkg_version('huggingface-hub', 'huggingface_hub'),
            'unsloth': pkg_version('unsloth'),
            'unsloth-zoo': pkg_version('unsloth-zoo', 'unsloth_zoo'),
            'torchao': pkg_version('torchao'),
        }

        is_p100 = capability == (6, 0)
        torch_arch_ok = ('sm_60' in arch) if is_p100 else True

        print('GPU:', gpu_name)
        print('GPU capability:', capability)
        print('Torch CUDA runtime:', torch.version.cuda)
        print('Torch CUDA arch list:', arch)
        for k, v in versions.items():
            print(f'{k}:', v)

        return (
            versions['torch'] is not None and versions['torch'].startswith(PINNED['torch'])
            and torch_arch_ok
            and versions['transformers'] == PINNED['transformers']
            and versions['tokenizers'] == PINNED['tokenizers']
            and versions['trl'] == PINNED['trl']
            and versions['peft'] == PINNED['peft']
            and versions['datasets'] == PINNED['datasets']
            and hub_ok(versions['huggingface-hub'])
            and versions['unsloth'] is not None
            and versions['unsloth-zoo'] is not None
            and versions['torchao'] is None
        )
    except Exception as exc:
        print('Runtime validation failed:', repr(exc))
        return False


if MARKER.exists() and active_runtime_ok():
    print('Dependency check passed.')
else:
    if MARKER.exists():
        print('Marker exists, but runtime is not compatible. Reinstalling.')
    else:
        print('Installing MedGemma/Unsloth QLoRA stack for v33.')

    already_loaded = [m for m in CORE_MODULES if m in sys.modules]
    if already_loaded:
        print('Already loaded in this kernel:', ', '.join(already_loaded))
        print('Installing now, then restarting so Python reloads packages cleanly.')

    # Remove any stale TorchAO first; it can break importing transformers with Torch 2.6.0.
    try:
        pip_install([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
    except Exception as exc:
        print('Initial torchao uninstall failed or torchao was not installed; continuing:', repr(exc))

    # Install a CUDA 12.4 Torch 2.6 stack. Torch 2.5 reaches Gemma 3 training but
    # fails in transformers.masking_utils because Gemma 3 uses mask combinators
    # that require torch>=2.6. Torch 2.11 is still avoided because Unsloth pins <2.11.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
        'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0',
        '--index-url', 'https://download.pytorch.org/whl/cu124'
    ])

    # Scientific stack. --no-deps prevents pip from upgrading NumPy/Pandas behind our back.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
        'numpy==1.26.4', 'pandas==2.2.2', 'matplotlib==3.8.4', 'pillow==11.3.0', 'tqdm'
    ])

    # Hugging Face + training stack. These versions align with Unsloth's current Gemma 3 notebook:
    # transformers 4.56.2 requires tokenizers >=0.22,<0.23; pin tokenizers explicitly.
    # Keep hub < 1.0 for transformers 4.x.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
        'transformers==4.56.2',
        'tokenizers==0.22.1',
        'trl==0.22.2',
        'peft==0.18.0',
        'datasets==4.3.0',
        'huggingface-hub==0.36.0',
        'accelerate==1.9.0',
        'bitsandbytes==0.46.1',
        'sentencepiece', 'protobuf', 'safetensors', 'packaging', 'hf_xet>=1.1.0,<2.0',
        'hf_transfer', 'tyro', 'msgspec', 'cut_cross_entropy'
    ])

    # Unsloth packages without dependency resolution so they cannot pull a newer Torch.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
        'unsloth', 'unsloth_zoo'
    ])

    # Match xFormers to torch 2.6.0. If this fails, continue; eager attention is used for SigLIP.
    try:
        pip_install([
            sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
            'xformers==0.0.29.post3'
        ])
    except Exception as exc:
        print('xformers install failed; continuing without xformers:', repr(exc))

    # TorchAO is intentionally removed. This notebook uses bitsandbytes 4-bit loading,
    # not TorchAO quantization, so leaving torchao installed only adds another failure mode.
    try:
        pip_install([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
    except Exception as exc:
        print('torchao uninstall failed or torchao was not installed; continuing:', repr(exc))

    MARKER.write_text('installed')
    print('\nDependency repair complete. Restarting kernel now.')
    print('After Kaggle restarts, run the notebook from the top again.')
    os._exit(0)


## 2. Imports, constants, metrics helpers


In [ ]:
import os
import re
import json
import time
import math
import shutil
import zipfile
import random
import urllib.request
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Important: set before importing huggingface_hub / transformers / unsloth.
# Kaggle may set this to 1 globally, but hf_transfer is optional and often absent.
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
# Disable Unsloth/TorchDynamo compilation for this Kaggle Gemma3 vision run.
# This avoids ArgsMismatchError in unsloth_zoo temporary Gemma patches.
os.environ['UNSLOTH_COMPILE_DISABLE'] = '1'
os.environ['UNSLOTH_COMPILE_IGNORE_ERRORS'] = '1'
os.environ['UNSLOTH_FULLGRAPH'] = '0'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['TORCH_COMPILE_DISABLE'] = '1'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:64')

import numpy as np
import pandas as pd
import torch

# Extra guard: even with env vars, make TorchDynamo non-fatal if an imported library touches it.
try:
    import torch._dynamo as _torch_dynamo
    _torch_dynamo.config.suppress_errors = True
    try:
        _torch_dynamo.disable()
    except Exception:
        pass
    print('TorchDynamo disabled/suppressed for stable Kaggle training.')
except Exception as exc:
    print('TorchDynamo guard unavailable:', repr(exc))
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

# TorchAO is intentionally absent for this Kaggle Torch 2.5.1 stack.
# If it is still installed, transformers may crash while importing torchao before Unsloth starts.
try:
    from importlib import metadata as _guard_metadata
    _torchao_version = _guard_metadata.version('torchao')
    raise RuntimeError(
        f'torchao=={_torchao_version} is installed, but this notebook intentionally removes torchao. '
        'Run the dependency bootstrap cell first, allow the restart, then run from the top.'
    )
except _guard_metadata.PackageNotFoundError:
    print('TorchAO: not installed (intentional for this Kaggle Torch 2.5.1 stack)')

# Import Unsloth before datasets/transformers so its patches are applied.
from unsloth import FastVisionModel, FastModel, get_chat_template

from huggingface_hub import login, hf_hub_download, whoami
from datasets import load_dataset
from transformers import AutoConfig

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU compute capability:', torch.cuda.get_device_capability(0))
    print('Torch:', torch.__version__)
    print('Torch arch list:', torch.cuda.get_arch_list())
    if torch.cuda.get_device_capability(0) == (6, 0) and 'sm_60' not in torch.cuda.get_arch_list():
        raise RuntimeError('Active torch does not include sm_60 kernels. Rerun the dependency cell and restart.')


try:
    from importlib import metadata as importlib_metadata
    print('Transformers:', importlib_metadata.version('transformers'))
    print('Tokenizers:', importlib_metadata.version('tokenizers'))
    print('Hugging Face Hub:', importlib_metadata.version('huggingface-hub'))
    print('Datasets:', importlib_metadata.version('datasets'))
    print('Unsloth:', importlib_metadata.version('unsloth'))
    print('Unsloth Zoo:', importlib_metadata.version('unsloth-zoo'))
    try:
        print('TorchAO:', importlib_metadata.version('torchao'))
    except importlib_metadata.PackageNotFoundError:
        print('TorchAO: not installed')
except Exception as exc:
    print('Package version diagnostics unavailable:', repr(exc))

PROJECT_DIR = Path('/kaggle/working/MedGemma_Unsloth_QLoRA_CRC_lowmem_v33')
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
ADAPTER_DIR = PROJECT_DIR / 'medgemma_crc_unsloth_lora'
for p in [PROJECT_DIR, DATA_DIR, RESULTS_DIR, ADAPTER_DIR]:
    p.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
LABEL_DESCRIPTIONS = {
    'ADI': 'adipose tissue / fat',
    'BACK': 'empty background / no tissue',
    'DEB': 'debris / necrotic debris',
    'LYM': 'lymphocytes / lymphoid cells',
    'MUC': 'mucus / mucin pools',
    'MUS': 'smooth muscle',
    'NORM': 'normal colon mucosa',
    'STR': 'cancer-associated stroma',
    'TUM': 'colorectal adenocarcinoma epithelium / tumor epithelium',
}
LABEL_TO_DIGIT = {label: str(i) for i, label in enumerate(LABEL_NAMES)}
DIGIT_TO_LABEL = {str(i): label for i, label in enumerate(LABEL_NAMES)}

CLASS_INSTRUCTION = """Classify this H&E colorectal histology image patch.
Return exactly one digit and nothing else.

0 = ADI, adipose tissue / fat
1 = BACK, empty background / no tissue
2 = DEB, debris / necrotic debris
3 = LYM, lymphocytes / lymphoid cells
4 = MUC, mucus / mucin pools
5 = MUS, smooth muscle
6 = NORM, normal colon mucosa
7 = STR, cancer-associated stroma
8 = TUM, colorectal adenocarcinoma epithelium / tumor epithelium

Answer with only one digit from 0 to 8."""


def simple_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred)) if len(y_true) else 0.0


def simple_confusion_matrix(y_true, y_pred, labels):
    label_to_pos = {label: idx for idx, label in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)
    for t, p in zip(y_true, y_pred):
        if t in label_to_pos and p in label_to_pos:
            cm[label_to_pos[t], label_to_pos[p]] += 1
    return cm


def classification_report_df(y_true, y_pred, labels):
    cm = simple_confusion_matrix(y_true, y_pred, labels)
    rows = []
    total = cm.sum()
    correct = np.trace(cm)
    for i, label in enumerate(labels):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        support = cm[i, :].sum()
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({'label': label, 'precision': precision, 'recall': recall, 'f1-score': f1, 'support': int(support)})
    macro_f1 = float(np.mean([r['f1-score'] for r in rows])) if rows else 0.0
    weighted_f1 = float(sum(r['f1-score'] * r['support'] for r in rows) / total) if total else 0.0
    acc = float(correct / total) if total else 0.0
    rows.append({'label': 'accuracy', 'precision': acc, 'recall': acc, 'f1-score': acc, 'support': int(total)})
    rows.append({'label': 'macro avg', 'precision': np.mean([r['precision'] for r in rows[:len(labels)]]), 'recall': np.mean([r['recall'] for r in rows[:len(labels)]]), 'f1-score': macro_f1, 'support': int(total)})
    rows.append({'label': 'weighted avg', 'precision': 0.0, 'recall': 0.0, 'f1-score': weighted_f1, 'support': int(total)})
    return pd.DataFrame(rows)


def plot_confusion_matrix(cm, labels, title, output_path):
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation='nearest')
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=labels,
        yticklabels=labels,
        ylabel='True label',
        xlabel='Predicted label',
        title=title,
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
    threshold = cm.max() / 2 if cm.size and cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center')
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 3. Hugging Face login from Kaggle Secrets


In [ ]:
def get_hf_token():
    token = os.environ.get('HF_TOKEN', '').strip()
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
        if token:
            return token.strip()
    except Exception:
        pass
    print(
        'HF_TOKEN not found. Continuing anonymously for the public Unsloth mirror. '        'If you switch back to google/medgemma-4b-it, add a Kaggle secret named HF_TOKEN '        'from the same Hugging Face account that accepted the gated model terms.'
    )
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    try:
        print('Hugging Face account:', whoami(token=HF_TOKEN).get('name'))
    except Exception as exc:
        print('HF login worked, but whoami failed:', repr(exc))
else:
    print('No Hugging Face token in this runtime.')


## 4. Locate or download NCT-CRC-HE-100K and CRC-VAL-HE-7K


In [ ]:
NCT_CRC_HE_100K_URL = 'https://zenodo.org/records/1214456/files/NCT-CRC-HE-100K.zip'
CRC_VAL_HE_7K_URL = 'https://zenodo.org/records/1214456/files/CRC-VAL-HE-7K.zip'


def looks_like_crc_folder(path: Path) -> bool:
    return path.is_dir() and all((path / label).is_dir() for label in LABEL_NAMES)


def find_crc_folder(folder_name: str):
    roots = [Path('/kaggle/input'), DATA_DIR, Path('/kaggle/working')]
    for root in roots:
        if not root.exists():
            continue
        for candidate in root.glob(f'**/{folder_name}'):
            if looks_like_crc_folder(candidate):
                return candidate
    # Sometimes Kaggle inputs have class folders directly under a dataset folder.
    if folder_name == 'NCT-CRC-HE-100K':
        for candidate in Path('/kaggle/input').glob('*') if Path('/kaggle/input').exists() else []:
            if looks_like_crc_folder(candidate) and 'VAL' not in candidate.name.upper():
                return candidate
    return None


def download_and_extract(url, zip_name, expected_folder):
    existing = find_crc_folder(expected_folder)
    if existing is not None:
        print(f'Found {expected_folder}:', existing)
        return existing

    zip_path = DATA_DIR / zip_name
    if not zip_path.exists():
        print('Downloading:', url)
        print('This may take several minutes for NCT-CRC-HE-100K.')
        urllib.request.urlretrieve(url, zip_path)
        print('Downloaded:', zip_path)
    else:
        print('Using existing zip:', zip_path)

    print('Extracting:', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)

    extracted = find_crc_folder(expected_folder)
    if extracted is None:
        raise FileNotFoundError(f'Could not locate extracted {expected_folder}. Check zip layout under {DATA_DIR}.')
    print(f'Found {expected_folder}:', extracted)
    return extracted

TRAIN_DIR = download_and_extract(NCT_CRC_HE_100K_URL, 'NCT-CRC-HE-100K.zip', 'NCT-CRC-HE-100K')
TEST_DIR = download_and_extract(CRC_VAL_HE_7K_URL, 'CRC-VAL-HE-7K.zip', 'CRC-VAL-HE-7K')

train_all = load_dataset('imagefolder', data_dir=str(TRAIN_DIR), split='train')
test_dataset = load_dataset('imagefolder', data_dir=str(TEST_DIR), split='train')

train_label_names = train_all.features['label'].names
test_label_names = test_dataset.features['label'].names
print('Train samples:', len(train_all), train_label_names)
print('Test samples:', len(test_dataset), test_label_names)

if train_label_names != LABEL_NAMES:
    print('Warning: dataset label order differs from expected LABEL_NAMES.')
if test_label_names != LABEL_NAMES:
    print('Warning: test label order differs from expected LABEL_NAMES.')


## 5. Training configuration


In [ ]:
# Low-memory Kaggle P100/T4 settings.
# This run is intended to avoid OOM first. Once it works, increase TRAIN_PER_CLASS and MAX_STEPS.

# Default to Unsloth's 4-bit MedGemma repo instead of the gated Google repo.
# The Google repo can list files publicly, but actual downloads can fail in Kaggle if token access is not approved.
MEDGEMMA_MODEL_ID = 'unsloth/medgemma-4b-it-unsloth-bnb-4bit'
FALLBACK_MEDGEMMA_MODEL_ID = 'unsloth/medgemma-4b-it-bnb-4bit'
GOOGLE_GATED_MEDGEMMA_MODEL_ID = 'google/medgemma-4b-it'

# v33 forward diagnostics. Start with lightweight Unsloth modes that isolate the 4-bit repo/loading path.
# Options:
#   '4bit_unsloth_bnb'          -> QLoRA-style 4-bit Unsloth repo, the mode that failed.
#   '4bit_unsloth_bnb_no_flag'  -> same 4-bit repo, but do NOT pass load_in_4bit=True.
#   '4bit_bnb_fallback'         -> alternate Unsloth bnb-4bit repo.
#   '8bit_full_repo'            -> non-4-bit Unsloth repo loaded with load_in_8bit=True.
#   '16bit_full_repo'           -> non-4-bit Unsloth repo loaded in 16-bit; may OOM on T4.
# Run one or two modes at a time to avoid VRAM/cache confusion.
DIAGNOSTIC_MODES_TO_RUN = ['4bit_unsloth_bnb_no_flag', '4bit_bnb_fallback']
MEDGEMMA_FULL_UNSLOTH_REPO = 'unsloth/medgemma-4b-it'
RUN_BASE_PREFLIGHT = True
RUN_LORA_PREFLIGHT = True
RUN_TEXT_ONLY_PREFLIGHT = True
RUN_MULTIMODAL_PREFLIGHT = True
RUN_FOR_INFERENCE_BEFORE_PREFLIGHT = True
RUN_FOR_TRAINING_BEFORE_PREFLIGHT = True
SAVE_DIAGNOSTIC_JSON = True

# If you intentionally want to retry the gated Google repo, set:
# MEDGEMMA_MODEL_ID = GOOGLE_GATED_MEDGEMMA_MODEL_ID

# Start conservative on P100/T4 16GB.
TRAIN_PER_CLASS = 1             # v33 debug only: one sample per class is enough for forward diagnostics.
MAX_STEPS = 0                   # v33 debug notebook does not train by default.
NUM_TRAIN_EPOCHS = 1            # Used only if MAX_STEPS is None.
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8 # Effective batch 8 without increasing VRAM.
LEARNING_RATE = 1e-5            # v33: lower LR to avoid NaN loss on Gemma3/MedGemma QLoRA.
LORA_R = 8                      # Lower VRAM than r=16/32.
LORA_ALPHA = 16
SAVE_STEPS = 25                 # Frequent checkpoints; v33 uses a stability smoke run.
LOGGING_STEPS = 1
MAX_SEQUENCE_LENGTH = 768       # Lower than 2048 to reduce activation memory.

# Gemma 3 / MedGemma vision dtype controls.
# Default fix for the Half/Float LayerNorm crash: do NOT upcast only LayerNorms.
# Instead, feed SigLIP pixel_values in the same dtype as the vision tower and disable
# gradient checkpointing inside the frozen/non-primary vision tower.
FINETUNE_VISION_LAYERS = False          # safer on Kaggle T4; language/projector LoRA still learns from image features.
CAST_PIXEL_VALUES_TO_VISION_DTYPE = False
DISABLE_VISION_GRADIENT_CHECKPOINTING = True
FORCE_FULL_VISION_FLOAT32 = True        # v33: required because v28 had non-finite digit logits in fp16 vision path.
# v33 feeds pixel_values as float32 and casts the full SigLIP vision tower + projector to float32.
PATCH_SIGLIP_LAYERNORM_SAFE_FP32 = True # v33: still safe, but full vision path is now float32.
PATCH_FOR_TRAINING_REAPPLY_VISION_FIXES = True # trainer calls model.for_training(); re-apply vision fixes there.
DISABLE_UNSLOTH_COMPILE = True           # v33: avoid TorchDynamo ArgsMismatchError in Unsloth Gemma q_norm compile path.
USE_GRADIENT_CHECKPOINTING = True        # Set False only if compiler-disable still fails; may use more VRAM.

# v33 NaN-loss stability settings. Start conservative, then scale up after finite loss.
MAX_GRAD_NORM = 0.05
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.0
OPTIMIZER_NAME = 'adamw_torch'       # safer than adamw_torch_fused for this patched Kaggle stack.
LR_SCHEDULER_TYPE = 'linear'
ABORT_ON_NONFINITE_LOSS = True
RUN_ONE_BATCH_LOSS_PREFLIGHT = True

# v33 label/loss safety: train only the final assistant digit, not the prompt.
# The default Unsloth vision collator can leave the whole chat prompt in labels.
# For classification, the only supervised token should be the answer digit.
TRAIN_ONLY_FINAL_ASSISTANT_DIGIT = True
USE_ACTIVE_ONLY_DIGIT_LOSS = True


# Evaluation. Start with 1000 for a fast sanity check. Set None for the full 7,180-image benchmark.
EVAL_MAX_SAMPLES = 1000
EVAL_CHECKPOINT_EVERY = 50

print({
    'MEDGEMMA_MODEL_ID': MEDGEMMA_MODEL_ID,
    'TRAIN_PER_CLASS': TRAIN_PER_CLASS,
    'MAX_STEPS': MAX_STEPS,
    'NUM_TRAIN_EPOCHS': NUM_TRAIN_EPOCHS,
    'EVAL_MAX_SAMPLES': EVAL_MAX_SAMPLES,
    'LORA_R': LORA_R,
    'GRADIENT_ACCUMULATION_STEPS': GRADIENT_ACCUMULATION_STEPS,
    'MAX_SEQUENCE_LENGTH': MAX_SEQUENCE_LENGTH,
    'FINETUNE_VISION_LAYERS': FINETUNE_VISION_LAYERS,
    'CAST_PIXEL_VALUES_TO_VISION_DTYPE': CAST_PIXEL_VALUES_TO_VISION_DTYPE,
    'DISABLE_VISION_GRADIENT_CHECKPOINTING': DISABLE_VISION_GRADIENT_CHECKPOINTING,
    'FORCE_FULL_VISION_FLOAT32': FORCE_FULL_VISION_FLOAT32,
    'PATCH_SIGLIP_LAYERNORM_SAFE_FP32': PATCH_SIGLIP_LAYERNORM_SAFE_FP32,
    'PATCH_FOR_TRAINING_REAPPLY_VISION_FIXES': PATCH_FOR_TRAINING_REAPPLY_VISION_FIXES,
    'DISABLE_UNSLOTH_COMPILE': DISABLE_UNSLOTH_COMPILE,
    'USE_GRADIENT_CHECKPOINTING': USE_GRADIENT_CHECKPOINTING,
    'MAX_GRAD_NORM': MAX_GRAD_NORM,
    'WARMUP_RATIO': WARMUP_RATIO,
    'WEIGHT_DECAY': WEIGHT_DECAY,
    'OPTIMIZER_NAME': OPTIMIZER_NAME,
    'LR_SCHEDULER_TYPE': LR_SCHEDULER_TYPE,
    'TRAIN_ONLY_FINAL_ASSISTANT_DIGIT': TRAIN_ONLY_FINAL_ASSISTANT_DIGIT,
    'USE_ACTIVE_ONLY_DIGIT_LOSS': USE_ACTIVE_ONLY_DIGIT_LOSS,
    'DIAGNOSTIC_MODES_TO_RUN': DIAGNOSTIC_MODES_TO_RUN,
    'MEDGEMMA_FULL_UNSLOTH_REPO': MEDGEMMA_FULL_UNSLOTH_REPO,
})


### v33 stability note

This version starts from the v26 working compile-disabled stack but treats `nan` training loss as numeric instability until proven otherwise. It lowers the LoRA learning rate, avoids fused AdamW, logs every step, checks that labels contain supervised tokens, runs a one-batch finite-loss preflight, and stops automatically if loss becomes non-finite.


## 6. Build a balanced training subset


In [ ]:
def label_name_for_sample(sample, feature_label_names):
    return feature_label_names[int(sample['label'])]


def balanced_indices(dataset, feature_label_names, per_class, seed=SEED):
    rng = random.Random(seed)
    buckets = {label: [] for label in LABEL_NAMES}
    for idx, sample in enumerate(dataset):
        label = label_name_for_sample(sample, feature_label_names)
        if label in buckets:
            buckets[label].append(idx)
    chosen = []
    for label in LABEL_NAMES:
        idxs = buckets[label]
        rng.shuffle(idxs)
        take = len(idxs) if per_class is None else min(per_class, len(idxs))
        chosen.extend(idxs[:take])
        print(f'{label}: selected {take} / {len(idxs)}')
    rng.shuffle(chosen)
    return chosen

print('Number of labels:', len(LABEL_NAMES))
print('Digit mapping:', DIGIT_TO_LABEL)
print('Note: target digit 7 means class STR, not 7 total labels. There are 9 classes: 0 through 8.')

train_indices = balanced_indices(train_all, train_label_names, TRAIN_PER_CLASS)
train_subset = train_all.select(train_indices)
print('Training subset size:', len(train_subset))

# Save selected indices for reproducibility.
pd.DataFrame({'train_index': train_indices}).to_csv(RESULTS_DIR / 'selected_train_indices.csv', index=False)


## 7. Conversation dataset with single-token digit targets


In [ ]:
def make_user_message(include_image_placeholder=True):
    content = [{'type': 'text', 'text': CLASS_INSTRUCTION}]
    if include_image_placeholder:
        content.append({'type': 'image'})
    return {'role': 'user', 'content': content}


def convert_sample_to_conversation(sample, feature_label_names):
    label = label_name_for_sample(sample, feature_label_names)
    digit = LABEL_TO_DIGIT[label]
    image = sample['image']
    if hasattr(image, 'convert'):
        image = image.convert('RGB')
    return {
        'messages': [
            {
                'role': 'user',
                'content': [
                    {'type': 'text', 'text': CLASS_INSTRUCTION},
                    {'type': 'image', 'image': image},
                ],
            },
            {'role': 'assistant', 'content': [{'type': 'text', 'text': digit}]},
        ]
    }


class CRCConversationDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, feature_label_names):
        self.hf_dataset = hf_dataset
        self.feature_label_names = feature_label_names

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        return convert_sample_to_conversation(self.hf_dataset[int(idx)], self.feature_label_names)


train_conversations = CRCConversationDataset(train_subset, train_label_names)
print('Example training record:')
example = train_conversations[0]
print(example['messages'][0]['content'][0]['text'][:300] + '...')
target_digit = example['messages'][1]['content'][0]['text']
target_label = DIGIT_TO_LABEL[target_digit]
print(f'target digit: {target_digit} -> {target_label}')
print('Reminder: labels are zero-indexed, so digit 7 is the 8th class, STR.')


## 8. Unsloth forward diagnostics: isolate non-finite logits

This notebook intentionally does **not** train by default. It tests whether Unsloth can produce finite logits before any optimizer step.

The default mode is `4bit_unsloth_bnb`, which is the QLoRA-style path that produced non-finite logits. After running it, restart the Kaggle kernel and change `DIAGNOSTIC_MODES_TO_RUN` to `['8bit_full_repo']` or `['16bit_full_repo']` to isolate whether the failure is specific to 4-bit quantization or to the Unsloth multimodal forward path itself.


In [ ]:

# v33: Unsloth-only forward diagnostics.
# Goal: prove which forward path first creates non-finite logits:
#   1) text-only forward
#   2) multimodal/image forward
#   3) before LoRA vs after LoRA
#   4) 4-bit vs 8-bit vs 16-bit Unsloth loading

from unsloth.trainer import UnslothVisionDataCollator
import torch.nn.functional as F
import traceback

# Re-assert compile guards before model loading.
os.environ['UNSLOTH_COMPILE_DISABLE'] = '1'
os.environ['UNSLOTH_COMPILE_IGNORE_ERRORS'] = '1'
os.environ['UNSLOTH_FULLGRAPH'] = '0'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['TORCH_COMPILE_DISABLE'] = '1'
try:
    import torch._dynamo as _torch_dynamo
    _torch_dynamo.config.suppress_errors = True
    try:
        _torch_dynamo.disable()
    except Exception:
        pass
    print('TorchDynamo disabled/suppressed for diagnostics.')
except Exception as exc:
    print('TorchDynamo guard unavailable:', repr(exc))


def clear_cuda(label=''):
    import gc
    if label:
        print('\n--- clearing CUDA:', label, '---')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('allocated GB:', round(torch.cuda.memory_allocated() / 1024**3, 3))
        print('reserved  GB:', round(torch.cuda.memory_reserved() / 1024**3, 3))


def verify_model_repo(repo_id, token=None):
    print('Checking model repo config:', repo_id)
    config_path = hf_hub_download(
        repo_id=repo_id,
        filename='config.json',
        token=token,
        force_download=False,
    )
    print('Config OK:', config_path)
    return config_path


def preview_eager_config(repo_id, token=None):
    cfg = AutoConfig.from_pretrained(
        repo_id,
        token=token,
        trust_remote_code=True,
    )
    cfg._attn_implementation = 'eager'
    if getattr(cfg, 'text_config', None) is not None:
        cfg.text_config._attn_implementation = 'eager'
    if getattr(cfg, 'vision_config', None) is not None:
        cfg.vision_config._attn_implementation = 'eager'
    print('Preview config attn:', getattr(cfg, '_attn_implementation', None))
    for attr in ('text_config', 'vision_config'):
        sub = getattr(cfg, attr, None)
        if sub is not None:
            print(f'Preview config.{attr} attn:', getattr(sub, '_attn_implementation', None))
    return cfg


def get_module_by_ending(model, ending):
    matches = [(name, module) for name, module in model.named_modules() if name.endswith(ending)]
    return matches[-1] if matches else (None, None)


def is_vision_module_name(name):
    lname = name.lower()
    return (
        'vision_tower' in lname
        or 'vision_model' in lname
        or 'siglip' in lname
        or '.vision.' in lname
    )


def disable_vision_gradient_checkpointing(model):
    disabled = []
    for name, module in model.named_modules():
        if is_vision_module_name(name):
            if hasattr(module, 'gradient_checkpointing'):
                try:
                    module.gradient_checkpointing = False
                    disabled.append(name)
                except Exception:
                    pass
            if hasattr(module, 'gradient_checkpointing_disable'):
                try:
                    module.gradient_checkpointing_disable()
                    disabled.append(name + '.gradient_checkpointing_disable()')
                except Exception:
                    pass
    print(f'Vision gradient checkpointing disabled on {len(disabled)} modules/hooks.')
    if disabled:
        print('First disabled:', disabled[0])
        print('Last disabled:', disabled[-1])
    return disabled


def patch_siglip_layernorm_safe_fp32(model):
    import types
    import torch.nn.functional as F
    patched = []
    already = []
    for name, module in model.named_modules():
        if is_vision_module_name(name) and isinstance(module, torch.nn.LayerNorm):
            if getattr(module, '_medgemma_safe_layernorm_patched', False):
                already.append(name)
                continue
            def _safe_forward(self, input):
                output_dtype = input.dtype
                weight = self.weight.float() if self.weight is not None else None
                bias = self.bias.float() if self.bias is not None else None
                output = F.layer_norm(input.float(), self.normalized_shape, weight, bias, self.eps)
                return output.to(dtype=output_dtype)
            safe_forward = types.MethodType(_safe_forward, module)
            if hasattr(module, '_old_forward'):
                module._old_forward = safe_forward
            else:
                module.forward = safe_forward
            module._medgemma_safe_layernorm_patched = True
            patched.append(name)
    print(f'SigLIP safe LayerNorm patch: {len(patched)} newly patched; {len(already)} already patched.')
    return patched + already


def force_full_vision_path_float32(model):
    target_endings = ('vision_tower', 'multi_modal_projector')
    patched = []
    for ending in target_endings:
        name, module = get_module_by_ending(model, ending)
        if module is not None:
            module.to(torch.float32)
            patched.append(name)
    print(f'Full vision/projection float32 patch: {len(patched)} top-level modules cast.')
    for name in patched:
        print('  cast:', name)
    return patched


def apply_vision_runtime_fixes(model, where='manual'):
    print(f'Applying vision runtime fixes from: {where}')
    if PATCH_SIGLIP_LAYERNORM_SAFE_FP32:
        patch_siglip_layernorm_safe_fp32(model)
    if DISABLE_VISION_GRADIENT_CHECKPOINTING:
        disable_vision_gradient_checkpointing(model)
    if FORCE_FULL_VISION_FLOAT32:
        force_full_vision_path_float32(model)
    return model


def infer_vision_tower_dtype(model):
    name, module = get_module_by_ending(model, 'vision_tower')
    if module is None:
        for n, m in model.named_modules():
            if 'vision_tower' in n.lower() or 'vision_model' in n.lower() or 'siglip' in n.lower():
                name, module = n, m
                break
    if module is None:
        print('Could not find vision tower; defaulting pixel dtype to float32.')
        return torch.float32
    for sub_name, sub_module in module.named_modules():
        if isinstance(sub_module, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.Embedding)):
            for p in sub_module.parameters(recurse=False):
                if torch.is_floating_point(p):
                    print('Vision tower dtype source:', f'{name}.{sub_name}', p.dtype)
                    return p.dtype
    for p in module.parameters():
        if torch.is_floating_point(p):
            print('Vision tower dtype source:', name, p.dtype)
            return p.dtype
    print('Vision tower has no floating parameters; defaulting pixel dtype to float32.')
    return torch.float32


def extract_assistant_digit(example):
    messages = example.get('messages', [])
    for message in reversed(messages):
        if message.get('role') != 'assistant':
            continue
        content = message.get('content', '')
        pieces = []
        if isinstance(content, list):
            for item in content:
                if isinstance(item, dict) and item.get('type') == 'text':
                    pieces.append(str(item.get('text', '')))
                elif isinstance(item, str):
                    pieces.append(item)
        else:
            pieces.append(str(content))
        text = ''.join(pieces).strip()
        match = re.search(r'[0-8]', text)
        if match:
            return match.group(0)
    raise RuntimeError(f'Could not find assistant digit in example: {example}')


def find_last_subsequence(sequence, subsequence):
    if not subsequence or len(subsequence) > len(sequence):
        return -1
    last_start = len(sequence) - len(subsequence)
    for start in range(last_start, -1, -1):
        if sequence[start:start + len(subsequence)] == subsequence:
            return start
    return -1


def candidate_digit_token_sequences(tokenizer, digit):
    candidates = []
    for text in [digit, '\n' + digit, ' ' + digit, digit + '<end_of_turn>']:
        ids = tokenizer(text, add_special_tokens=False).input_ids
        if ids and ids not in candidates:
            candidates.append(ids)
    return candidates


class SafeDigitOnlyGemma3VisionDataCollator:
    def __init__(self, base_collator, tokenizer, target_pixel_dtype=None, digit_only=True):
        self.base_collator = base_collator
        self.tokenizer = tokenizer
        self.target_pixel_dtype = target_pixel_dtype
        self.digit_only = digit_only

    def _apply_digit_only_labels(self, batch, examples):
        if not self.digit_only:
            return batch
        input_ids = batch.get('input_ids')
        if input_ids is None:
            raise RuntimeError('Collator batch has no input_ids; cannot build digit-only labels.')
        new_labels = torch.full_like(input_ids, fill_value=-100)
        debug_rows = []
        for row_idx, example in enumerate(examples):
            digit = extract_assistant_digit(example)
            token_sequences = candidate_digit_token_sequences(self.tokenizer, digit)
            seq = input_ids[row_idx].detach().cpu().tolist()
            matches = []
            for ids in token_sequences:
                start = find_last_subsequence(seq, ids)
                if start >= 0:
                    matches.append((start, len(ids), ids))
            if not matches:
                tail_ids = seq[-80:]
                tail_text = self.tokenizer.decode(tail_ids, skip_special_tokens=False)
                raise RuntimeError(
                    f'Could not locate assistant digit {digit!r} in tokenized row {row_idx}. '
                    f'Tail decode: {tail_text!r}'
                )
            start, span_len, ids = max(matches, key=lambda item: (item[0], -item[1]))
            new_labels[row_idx, start:start + span_len] = input_ids[row_idx, start:start + span_len]
            debug_rows.append({
                'row': row_idx,
                'digit': digit,
                'start': int(start),
                'span_len': int(span_len),
                'ids': [int(x) for x in ids],
                'decoded': self.tokenizer.decode(ids, skip_special_tokens=False),
            })
        batch['labels'] = new_labels
        batch['_digit_label_debug'] = debug_rows
        return batch

    def __call__(self, examples):
        batch = self.base_collator(examples)
        if self.target_pixel_dtype is not None and 'pixel_values' in batch:
            pixel_values = batch['pixel_values']
            if torch.is_tensor(pixel_values) and torch.is_floating_point(pixel_values):
                batch['pixel_values'] = pixel_values.to(dtype=self.target_pixel_dtype)
        batch = self._apply_digit_only_labels(batch, examples)
        return batch


def move_batch_to_device(batch, device):
    out = {}
    for k, v in batch.items():
        if k.startswith('_'):
            continue
        if torch.is_tensor(v):
            out[k] = v.to(device)
        else:
            out[k] = v
    return out


def tensor_stats(tensor, name):
    t = tensor.detach()
    finite = torch.isfinite(t)
    total = int(t.numel())
    finite_count = int(finite.sum().detach().cpu().item()) if total else 0
    nan_count = int(torch.isnan(t).sum().detach().cpu().item()) if total else 0
    posinf_count = int(torch.isposinf(t).sum().detach().cpu().item()) if total else 0
    neginf_count = int(torch.isneginf(t).sum().detach().cpu().item()) if total else 0
    stats = {
        'name': name,
        'shape': tuple(t.shape),
        'dtype': str(t.dtype),
        'device': str(t.device),
        'finite_count': finite_count,
        'nan_count': nan_count,
        'posinf_count': posinf_count,
        'neginf_count': neginf_count,
        'total': total,
        'all_finite': bool(finite_count == total),
    }
    if finite_count:
        vals = t[finite].float()
        stats['min'] = float(vals.min().detach().cpu())
        stats['max'] = float(vals.max().detach().cpu())
        stats['mean'] = float(vals.mean().detach().cpu())
    else:
        stats['min'] = None
        stats['max'] = None
        stats['mean'] = None
    return stats


def print_stats(stats):
    print(
        f"{stats['name']}: shape={stats['shape']}, dtype={stats['dtype']}, "
        f"finite={stats['finite_count']}/{stats['total']}, all_finite={stats['all_finite']}, "
        f"nan={stats.get('nan_count')}, +inf={stats.get('posinf_count')}, -inf={stats.get('neginf_count')}, "
        f"min={stats['min']}, max={stats['max']}"
    )


def text_only_forward_check(model, processor):
    tokenizer = processor.tokenizer if hasattr(processor, 'tokenizer') else processor
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': 'Return exactly one digit and nothing else: 7'}]}]
    try:
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        try:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            text = '<bos><start_of_turn>user\nReturn exactly one digit and nothing else: 7<end_of_turn>\n<start_of_turn>model\n'
    inputs = tokenizer(text, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items() if torch.is_tensor(v)}
    with torch.no_grad():
        out = model(**inputs, use_cache=False, return_dict=True)
    stats = tensor_stats(out.logits, 'text_only_logits')
    print_stats(stats)
    return stats


def multimodal_digit_forward_check(model, processor, example, target_pixel_dtype=None, label='multimodal'):
    base_collator = UnslothVisionDataCollator(model, processor)
    collator = SafeDigitOnlyGemma3VisionDataCollator(
        base_collator,
        tokenizer=processor.tokenizer,
        target_pixel_dtype=target_pixel_dtype,
        digit_only=True,
    )
    batch = collator([example])
    debug_info = batch.pop('_digit_label_debug', None)
    print('Digit label debug:', debug_info)
    for key, value in batch.items():
        if torch.is_tensor(value):
            print(f'  {key}: shape={tuple(value.shape)}, dtype={value.dtype}, device={value.device}')
    labels = batch['labels']
    supervised = labels[0][labels[0] != -100]
    print('Supervised label tokens:', int(supervised.numel()))
    if int(supervised.numel()) > 0:
        print('Decoded supervised target:', repr(processor.tokenizer.decode(supervised.detach().cpu().tolist(), skip_special_tokens=False)))
    inputs = move_batch_to_device(batch, model.device)
    labels = inputs.pop('labels')
    with torch.no_grad():
        out = model(**inputs, use_cache=False, return_dict=True)
    logits = out.logits
    stats_all = tensor_stats(logits, f'{label}_all_logits')
    print_stats(stats_all)
    shift_logits = logits[..., :-1, :]
    shift_labels = labels[..., 1:]
    active = shift_labels != -100
    active_count = int(active.sum().detach().cpu().item())
    print('Active shifted digit labels:', active_count)
    if active_count == 0:
        raise RuntimeError('No active digit labels after shift; cannot compute active loss.')
    active_logits = shift_logits[active].float()
    active_labels = shift_labels[active].to(active_logits.device)
    stats_active = tensor_stats(active_logits, f'{label}_active_digit_logits')
    print_stats(stats_active)
    loss = None
    if stats_active['all_finite']:
        loss = F.cross_entropy(active_logits, active_labels, reduction='mean')
        print(f'{label} active-digit loss:', float(loss.detach().cpu()))
    else:
        print(f'{label} active-digit loss skipped because active logits are non-finite.')
    return {
        'debug_info': debug_info,
        'all_logits': stats_all,
        'active_digit_logits': stats_active,
        'loss': None if loss is None else float(loss.detach().cpu()),
    }


def get_mode_config(mode_name):
    if mode_name == '4bit_unsloth_bnb':
        return {
            'repo': MEDGEMMA_MODEL_ID,
            'fallback_repo': FALLBACK_MEDGEMMA_MODEL_ID,
            'load_kwargs': dict(load_in_4bit=True, load_in_8bit=False, load_in_16bit=False),
            'description': '4-bit Unsloth bnb repo, QLoRA-style path',
        }
    if mode_name == '4bit_unsloth_bnb_no_flag':
        return {
            'repo': MEDGEMMA_MODEL_ID,
            'fallback_repo': FALLBACK_MEDGEMMA_MODEL_ID,
            'load_kwargs': dict(load_in_4bit=False, load_in_8bit=False, load_in_16bit=False),
            'description': 'Same pre-quantized Unsloth bnb repo, but do not request load_in_4bit again',
        }
    if mode_name == '4bit_bnb_fallback':
        return {
            'repo': FALLBACK_MEDGEMMA_MODEL_ID,
            'fallback_repo': MEDGEMMA_MODEL_ID,
            'load_kwargs': dict(load_in_4bit=True, load_in_8bit=False, load_in_16bit=False),
            'description': 'Alternate Unsloth bnb-4bit repo with load_in_4bit=True',
        }
    if mode_name == '4bit_bnb_fallback_no_flag':
        return {
            'repo': FALLBACK_MEDGEMMA_MODEL_ID,
            'fallback_repo': MEDGEMMA_MODEL_ID,
            'load_kwargs': dict(load_in_4bit=False, load_in_8bit=False, load_in_16bit=False),
            'description': 'Alternate Unsloth bnb-4bit repo without load_in_4bit flag',
        }
    if mode_name == '8bit_full_repo':
        return {
            'repo': MEDGEMMA_FULL_UNSLOTH_REPO,
            'fallback_repo': GOOGLE_GATED_MEDGEMMA_MODEL_ID,
            'load_kwargs': dict(load_in_4bit=False, load_in_8bit=True, load_in_16bit=False),
            'description': 'Full Unsloth repo loaded with 8-bit quantization',
        }
    if mode_name == '16bit_full_repo':
        return {
            'repo': MEDGEMMA_FULL_UNSLOTH_REPO,
            'fallback_repo': GOOGLE_GATED_MEDGEMMA_MODEL_ID,
            'load_kwargs': dict(load_in_4bit=False, load_in_8bit=False, load_in_16bit=True),
            'description': 'Full Unsloth repo loaded in 16-bit; may OOM on T4',
        }
    raise ValueError(f'Unknown diagnostic mode: {mode_name}')


def load_unsloth_model_for_mode(mode_name):
    cfg = get_mode_config(mode_name)
    repo_candidates = []
    for repo_id in [cfg['repo'], cfg.get('fallback_repo')]:
        if repo_id and repo_id not in repo_candidates:
            repo_candidates.append(repo_id)
    repo_errors = []
    selected_repo = None
    for repo_id in repo_candidates:
        try:
            verify_model_repo(repo_id, HF_TOKEN)
            selected_repo = repo_id
            break
        except Exception as exc:
            repo_errors.append((repo_id, repr(exc)))
            print('Repo check failed:', repo_id, repr(exc))
    if selected_repo is None:
        raise RuntimeError(f'No repo worked for mode {mode_name}: {repo_errors}')
    preview_eager_config(selected_repo, HF_TOKEN)
    common = dict(
        model_name=selected_repo,
        max_seq_length=MAX_SEQUENCE_LENGTH,
        token=HF_TOKEN,
        trust_remote_code=True,
        device_map='auto',
        use_gradient_checkpointing='unsloth' if USE_GRADIENT_CHECKPOINTING else False,
        attn_implementation='eager',
        unsloth_force_compile=False,
    )
    common.update(cfg['load_kwargs'])
    print('\nLoading mode:', mode_name, '| repo:', selected_repo, '|', cfg['description'])
    print('load kwargs:', cfg['load_kwargs'])
    loader_name = 'FastVisionModel'
    try:
        loaded_model, loaded_processor = FastVisionModel.from_pretrained(**common)
        loader_api = FastVisionModel
    except Exception as first_exc:
        print('FastVisionModel load failed, retrying with FastModel:', repr(first_exc))
        loaded_model, loaded_processor = FastModel.from_pretrained(**common)
        loader_api = FastModel
        loader_name = 'FastModel'
    try:
        loaded_processor = get_chat_template(loaded_processor, 'gemma-3')
    except Exception as exc:
        print('get_chat_template failed; continuing:', repr(exc))
    print('Loaded with:', loader_name)
    return loaded_model, loaded_processor, loader_api, selected_repo, loader_name


def add_lora_adapters(loaded_model, loader_api):
    peft_kwargs = dict(
        finetune_vision_layers=FINETUNE_VISION_LAYERS,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=False,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0,
        bias='none',
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )
    try:
        lora_model = FastVisionModel.get_peft_model(loaded_model, **peft_kwargs)
    except Exception as first_exc:
        if hasattr(loader_api, 'get_peft_model') and loader_api is not FastVisionModel:
            print('FastVisionModel.get_peft_model failed. Retrying with loader API:', repr(first_exc))
            lora_model = loader_api.get_peft_model(loaded_model, **peft_kwargs)
        else:
            raise
    return lora_model


def run_mode_diagnostic(mode_name, example):
    result = {'mode': mode_name, 'ok': False, 'stages': {}}
    loaded_model = None
    loaded_processor = None
    try:
        clear_cuda(f'before {mode_name}')
        loaded_model, loaded_processor, loader_api, selected_repo, loader_name = load_unsloth_model_for_mode(mode_name)
        result.update({'repo': selected_repo, 'loader': loader_name})
        apply_vision_runtime_fixes(loaded_model, where=f'{mode_name}: base model')
        vision_dtype = infer_vision_tower_dtype(loaded_model)
        target_pixel_dtype = vision_dtype if CAST_PIXEL_VALUES_TO_VISION_DTYPE else None
        print('Data collator target pixel dtype:', target_pixel_dtype)
        if RUN_FOR_INFERENCE_BEFORE_PREFLIGHT and hasattr(loader_api, 'for_inference'):
            try:
                loader_api.for_inference(loaded_model)
                print('Called loader_api.for_inference(model) before base checks.')
            except Exception as exc:
                print('for_inference before base checks failed:', repr(exc))
        if RUN_FOR_TRAINING_BEFORE_PREFLIGHT and hasattr(loaded_model, 'for_training'):
            try:
                loaded_model.for_training(use_gradient_checkpointing=False)
                apply_vision_runtime_fixes(loaded_model, where=f'{mode_name}: after explicit for_training')
                print('Called model.for_training(use_gradient_checkpointing=False) before base checks.')
            except Exception as exc:
                print('for_training before base checks failed:', repr(exc))
        if RUN_TEXT_ONLY_PREFLIGHT:
            print('\n[base] text-only forward')
            result['stages']['base_text_only'] = text_only_forward_check(loaded_model, loaded_processor)
        if RUN_MULTIMODAL_PREFLIGHT:
            print('\n[base] multimodal digit forward')
            result['stages']['base_multimodal'] = multimodal_digit_forward_check(
                loaded_model, loaded_processor, example, target_pixel_dtype=target_pixel_dtype, label='base_multimodal'
            )
        if RUN_LORA_PREFLIGHT:
            print('\nAdding LoRA adapters...')
            loaded_model = add_lora_adapters(loaded_model, loader_api)
            apply_vision_runtime_fixes(loaded_model, where=f'{mode_name}: after LoRA')
            vision_dtype = infer_vision_tower_dtype(loaded_model)
            target_pixel_dtype = vision_dtype if CAST_PIXEL_VALUES_TO_VISION_DTYPE else None
            print('Data collator target pixel dtype after LoRA:', target_pixel_dtype)
            try:
                loaded_model.print_trainable_parameters()
            except Exception as exc:
                print('print_trainable_parameters unavailable:', repr(exc))
            if RUN_TEXT_ONLY_PREFLIGHT:
                print('\n[LoRA] text-only forward')
                result['stages']['lora_text_only'] = text_only_forward_check(loaded_model, loaded_processor)
            if RUN_MULTIMODAL_PREFLIGHT:
                print('\n[LoRA] multimodal digit forward')
                result['stages']['lora_multimodal'] = multimodal_digit_forward_check(
                    loaded_model, loaded_processor, example, target_pixel_dtype=target_pixel_dtype, label='lora_multimodal'
                )
        # ok means every stats dict that exists has finite active digit logits if present.
        result['ok'] = True
        for stage_name, stage in result['stages'].items():
            if isinstance(stage, dict) and 'active_digit_logits' in stage:
                if not stage['active_digit_logits']['all_finite']:
                    result['ok'] = False
            elif isinstance(stage, dict) and 'all_finite' in stage:
                if not stage['all_finite']:
                    result['ok'] = False
        print('\nMode result:', mode_name, 'ok =', result['ok'])
    except Exception as exc:
        result['ok'] = False
        result['error_type'] = type(exc).__name__
        result['error'] = repr(exc)
        result['traceback'] = traceback.format_exc()
        print('\nMode failed:', mode_name)
        print(type(exc).__name__, repr(exc))
        print(result['traceback'][-4000:])
    finally:
        try:
            del loaded_model, loaded_processor
        except Exception:
            pass
        clear_cuda(f'after {mode_name}')
    return result


# Choose a stable sample. train_conversations[0] is fine because the collator debug prints the true digit.
debug_example = train_conversations[0]
print('Diagnostic modes to run:', DIAGNOSTIC_MODES_TO_RUN)
print('Debug target digit:', extract_assistant_digit(debug_example))

all_results = []
for mode_name in DIAGNOSTIC_MODES_TO_RUN:
    all_results.append(run_mode_diagnostic(mode_name, debug_example))

print('\n=== DIAGNOSTIC SUMMARY ===')
for result in all_results:
    print(json.dumps({k: v for k, v in result.items() if k not in ('traceback', 'stages')}, indent=2))
    for stage_name, stage in result.get('stages', {}).items():
        if isinstance(stage, dict) and 'active_digit_logits' in stage:
            s = stage['active_digit_logits']
            print(f"  {stage_name}: active finite {s['finite_count']}/{s['total']} all_finite={s['all_finite']} loss={stage.get('loss')}")
        elif isinstance(stage, dict) and 'all_finite' in stage:
            print(f"  {stage_name}: finite {stage['finite_count']}/{stage['total']} all_finite={stage['all_finite']}")

if SAVE_DIAGNOSTIC_JSON:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    out_path = RESULTS_DIR / 'unsloth_forward_diagnostics_v33.json'
    with open(out_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print('Saved diagnostic JSON:', out_path)


## 9. How to interpret the diagnostic

- If `base_text_only` is non-finite, the Unsloth text path is broken for this package stack.
- If `base_text_only` is finite but `base_multimodal` is non-finite, the issue is the MedGemma/SigLIP image path.
- If base is finite but LoRA is non-finite, the LoRA injection target is causing the instability.
- If `4bit_unsloth_bnb` fails but `8bit_full_repo` or `16bit_full_repo` is finite, the failure is specific to the 4-bit QLoRA path.
- If all Unsloth modes fail before training, the practical fallback is plain Hugging Face PEFT/QLoRA or MedSigLIP + classifier.


## Final artifact packaging cell

Run this cell after diagnostics to download the JSON summary and small report files for submission or sharing.

In [ ]:
from pathlib import Path
import zipfile
import json
import os
from IPython.display import FileLink, display

WORKING = Path('/kaggle/working')
ZIP_PATH = WORKING / 'send_to_chatgpt_medgemma_unsloth_v34_stop_decision.zip'

candidate_dirs = [
    WORKING / 'MedGemma_Unsloth_QLoRA_CRC_lowmem_v33',
    WORKING / 'MedGemma_Unsloth_QLoRA_CRC_lowmem_v34',
    WORKING / 'MedGemma_Unsloth_Forward_Debug_v33',
    WORKING / 'MedGemma_Unsloth_Forward_Debug_v34',
]
small_exts = {'.json', '.csv', '.txt', '.log', '.md'}

def should_include(path: Path) -> bool:
    if not path.is_file():
        return False
    if path.suffix.lower() not in small_exts:
        return False
    if path.stat().st_size > 25 * 1024 * 1024:
        return False
    skip_terms = ['.cache', '__pycache__', 'wandb', '.safetensors', '.bin', '.pt', '.pth', '.arrow', '.parquet']
    return not any(term in str(path) for term in skip_terms)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

added = []
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for folder in candidate_dirs:
        if folder.exists():
            for p in folder.rglob('*'):
                if should_include(p):
                    arcname = p.relative_to(WORKING)
                    z.write(p, arcname)
                    added.append(str(arcname))
    for p in WORKING.glob('*'):
        if should_include(p):
            arcname = p.relative_to(WORKING)
            if str(arcname) not in added:
                z.write(p, arcname)
                added.append(str(arcname))
    manifest = {
        'purpose': 'MedGemma Unsloth stop-decision diagnostics',
        'zip_path': str(ZIP_PATH),
        'num_files': len(added),
        'files': sorted(added),
    }
    z.writestr('MANIFEST.json', json.dumps(manifest, indent=2))

print('Created:', ZIP_PATH)
print('Size MB:', round(ZIP_PATH.stat().st_size / (1024 * 1024), 3))
print('Files included:', len(added))
for f in sorted(added):
    print(' -', f)

os.chdir('/kaggle/working')
display(FileLink(ZIP_PATH.name))